In [13]:
import os 
import math 
import pathlib
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM ,AutoConfig
from datasets import load_dataset

from llmcompressor import oneshot 
from llmcompressor.modifiers.gptq import GPTQModifier

In [5]:
#download the Qwen3-0.6B model from hugging face and put it in folder named model

from pathlib import Path 
from huggingface_hub import snapshot_download 

MODEL_ID = "Qwen/Qwen3-0.6B"
MODEL_DIR = Path("models/Qwen3-0.6B")

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Downloading model...")


snapshot_download(
    repo_id = MODEL_ID,
    local_dir=str(MODEL_DIR),
)

print("Download Complete!")
print("Saved to:", MODEL_DIR.resolve())

Fetching 10 files: 100%|██████████| 10/10 [01:28<00:00,  8.82s/it]

Download Complete!
Saved to: C:\Users\Anil\Documents\Learning_Projects\local-llm-compression\models\Qwen3-0.6B


In [8]:
from pathlib import Path 
MODEL_DIR = Path("models/Qwen3-0.6B")
print(MODEL_DIR.exists())
print(list(MODEL_DIR.iterdir()))

True
[WindowsPath('models/Qwen3-0.6B/.cache'), WindowsPath('models/Qwen3-0.6B/.gitattributes'), WindowsPath('models/Qwen3-0.6B/config.json'), WindowsPath('models/Qwen3-0.6B/generation_config.json'), WindowsPath('models/Qwen3-0.6B/LICENSE'), WindowsPath('models/Qwen3-0.6B/merges.txt'), WindowsPath('models/Qwen3-0.6B/model.safetensors'), WindowsPath('models/Qwen3-0.6B/README.md'), WindowsPath('models/Qwen3-0.6B/tokenizer.json'), WindowsPath('models/Qwen3-0.6B/tokenizer_config.json'), WindowsPath('models/Qwen3-0.6B/vocab.json')]


In [21]:
config = AutoConfig.from_pretrained(MODEL_DIR)
#print the number of hidden layers
print(config.num_hidden_layers)
print(config.hidden_size)  #tells the dimensionality of the model's hidden representations
print(config.num_attention_heads) #tells the number of attention heads
print(config.head_dim) #tells the head dimension which is how many numerical features each attention head uses for q and k representations
print(config.num_key_value_heads)#how many key-value heads the model has 

#calucalte the GQA 
print(config.num_attention_heads // config.num_key_value_heads)

28
1024
16
128
8
2


In [28]:
text = "I am learning how large language model works"
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
tokens = tokenizer.tokenize(text)
print(tokens)

#tokens to id 
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print(token_ids)

#convert ids to PyTorch tensor
input_ids = torch.tensor([token_ids])

print(input_ids.shape)


['I', 'Ġam', 'Ġlearning', 'Ġhow', 'Ġlarge', 'Ġlanguage', 'Ġmodel', 'Ġworks']
[40, 1079, 6832, 1246, 3460, 4128, 1614, 4278]
torch.Size([1, 8])


In [ ]:
#load model into CPU 
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, torch_dtype=torch.bfloat16, device_map="cpu")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 694.76it/s]


In [ ]:
embedding_layer = model.get_input_embeddings()
print(embedding_layer.weight.shape) #this tells us that the vocab size is 151,936 and numerical value per embedding is 1024

torch.Size([151936, 1024])


In [ ]:
#lets perform the embedding lookup
embeddings = embedding_layer(input_ids)
print(embeddings.shape) #this tells us batch size of 1, sequence length of 8 and embedding dimension of 1024

torch.Size([1, 8, 1024])


In [34]:
#peek inside the first transformer block within the Qwen model 
print(model.model.layers[0])

Qwen3DecoderLayer(
  (self_attn): Qwen3Attention(
    (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
    (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
    (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
    (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
    (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
    (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
  )
  (mlp): Qwen3MLP(
    (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
    (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
    (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
  (post_attention_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
)


These are learned linear transformations.For example here q_proj transforms each token's
1024 dimensional representations into a 2048 dimensional query vector.
But why 2048?
We know that :
Attention heads = 16 
Head dimension = 128 
Therefore: 16 X 128 = 2048 which explain the output of q_proj

The key and value projections are smaller becuase Qwen3 uses Grouped Query Attentions(GQA) with
eight key/value heads. so 8 X 128 = 1024.



In [ ]:
#inspect the query projection weights
print(model.model.layers[0].self_attn.q_proj.weight.shape)
#linear-layer weights are as [out_features,in_features]

torch.Size([2048, 1024])


In [36]:
query_vectors = model.model.layers[0].self_attn.q_proj(embeddings)
print(query_vectors.shape)

torch.Size([1, 8, 2048])


In [ ]:
query_heads = query_vectors.reshape(1,8,16,128)
print(query_heads.shape) #Batch is 1, sequence length is 8, attention heads is 16 and features per head is 128

torch.Size([1, 8, 16, 128])


In [39]:
key_vectors = model.model.layers[0].self_attn.k_proj(embeddings)
print(key_vectors.shape)

torch.Size([1, 8, 1024])


In [40]:
key_heads = key_vectors.reshape(1,8,8,128)
print(key_heads.shape)

torch.Size([1, 8, 8, 128])


In [41]:
value_vectors = model.model.layers[0].self_attn.v_proj(embeddings)
print(value_vectors.shape)

torch.Size([1, 8, 1024])


In [42]:
value_heads = value_vectors.reshape(1,8,8,128)
print(value_heads.shape)

torch.Size([1, 8, 8, 128])


In [ ]:
#transpose the query tensor
query_heads = query_heads.transpose(1,2)
print(query_heads.shape)

torch.Size([1, 16, 8, 128])


In [44]:
#transpose the key tensor
key_heads = key_heads.transpose(1,2)
print(key_heads.shape)

torch.Size([1, 8, 8, 128])


In [45]:
#conceptually we want [Batch, Heads, Tokens, Features]
value_heads = value_heads.transpose(1,2)
print(value_heads.shape)

torch.Size([1, 8, 8, 128])


In [46]:
#To calculate attention using batched matrix multiplication we can repeat key head twice just to demonstrate attention
#real GQA doesn't need to repeat 
key_heads_expanded = key_heads.repeat_interleave(2,dim=1)
print(key_heads_expanded.shape)

torch.Size([1, 16, 8, 128])


In [50]:
#calculate the attention score 
attention_scores = torch.matmul(query_heads, key_heads_expanded.transpose(-2,-1))
print(attention_scores.shape)

scaled_scores = attention_scores / (128 ** 0.5)

torch.Size([1, 16, 8, 8])


In [52]:
#apply casual masking 
casual_mask = torch.tril(torch.ones(8,8,dtype=torch.bool))
print(casual_mask.shape)

masked_scores = scaled_scores.masked_fill(~casual_mask, float("-inf"))
print(masked_scores)


torch.Size([8, 8])
tensor([[[[ 2.1667e-03,        -inf,        -inf,  ...,        -inf,
                  -inf,        -inf],
          [ 1.7242e-03, -1.9455e-04,        -inf,  ...,        -inf,
                  -inf,        -inf],
          [ 2.7771e-03,  2.0447e-03,  5.9509e-04,  ...,        -inf,
                  -inf,        -inf],
          ...,
          [ 2.2125e-03,  3.0365e-03, -4.6939e-07,  ..., -2.0447e-03,
                  -inf,        -inf],
          [ 3.9368e-03,  2.5635e-03,  1.2398e-04,  ...,  7.5912e-04,
            1.7700e-03,        -inf],
          [ 2.5635e-03,  2.1667e-03,  2.9755e-03,  ...,  1.8215e-04,
            2.9755e-03,  2.7008e-03]],

         [[ 4.2114e-03,        -inf,        -inf,  ...,        -inf,
                  -inf,        -inf],
          [ 2.6093e-03,  3.2349e-03,        -inf,  ...,        -inf,
                  -inf,        -inf],
          [ 3.2196e-03,  2.3041e-03,  1.8997e-03,  ...,        -inf,
                  -inf,        -inf],
 

In [54]:
attention_weights = torch.softmax(masked_scores, dim =-1)
print(attention_weights[0,0,0,:])
print(attention_weights[0, 0, 1, :])


tensor([1., 0., 0., 0., 0., 0., 0., 0.], dtype=torch.bfloat16,
       grad_fn=<SelectBackward0>)
tensor([0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
       dtype=torch.bfloat16, grad_fn=<SelectBackward0>)


In [55]:
value_heads_expanded = value_heads.repeat_interleave(2,dim=1)
print(value_heads_expanded.shape)

torch.Size([1, 16, 8, 128])


In [56]:
attention_output = torch.matmul(attention_weights, value_heads_expanded)
print(attention_output.shape)

torch.Size([1, 16, 8, 128])


In [59]:
attention_output = attention_output.transpose(1,2)
print(attention_output.shape)

attention_output = attention_output.reshape(1,8,2048)
print(attention_output.shape)

torch.Size([1, 2048, 8])
torch.Size([1, 8, 2048])


In [61]:
projected_output = model.model.layers[0].self_attn.o_proj(attention_output)
print(projected_output.shape)
print(model.model.layers[0].self_attn.o_proj.weight.shape)

torch.Size([1, 8, 1024])
torch.Size([1024, 2048])


In [62]:
#count the parameters programatically
print(model.model.layers[0].self_attn.o_proj.weight.numel())

2097152


In [64]:
total_params = sum(p.numel() for p in model.parameters())
print(total_params)

596049920


In [68]:
#BF16 memory 
bf16_memory_gib = total_params * 2 /(1024 **3)
print(f"size of the model is {bf16_memory_gib} GB", )

size of the model is 1.1102294921875 GB
